In [1]:
from braket.program_sets import ProgramSet
from mpqp import QCircuit
from mpqp.core.languages import Language
from mpqp.gates import *
from mpqp.tools.circuit import random_circuit

cs = []

for _ in range(5):
    cs.append(random_circuit().to_other_language(Language.BRAKET))

s = ProgramSet(cs)

In [2]:
from braket.devices import LocalSimulator

device = LocalSimulator()
result_1 = device.run(s, shots=300).result()

In [3]:
for r in result_1:
    for ex in r:
        print(len(r))
        print(ex.counts)

1
Counter({'01000': 60})
1
Counter({'00000': 60})
1
Counter({'00000': 19, '00011': 19, '01011': 11, '01000': 11})
1
Counter({'10010': 55, '00010': 5})
1
Counter({'00101': 58, '00100': 2})


In [4]:
from sympy import Symbol
from braket.program_sets import CircuitBinding
from braket.devices import LocalSimulator

t = Symbol("t")
r = Symbol("r")

c1 = QCircuit([Rz(t, 0), Rx(r, 0), Rz(t, 0)]).to_other_language(Language.BRAKET)

cb = CircuitBinding(c1, input_sets={"t": (0, 1), "r": (1, 0.01)})
s1 = ProgramSet(cb, 100)
device = LocalSimulator()
result_2 = device.run(s1).result()
for r in result_2:
    for rr in r:
        print(rr.counts)

Counter({'0': 76, '1': 24})
Counter({'0': 100})


In [ ]:
from mpqp.core.circuit import BindingMode, CircuitBinding as MPQPBinding
from mpqp import QCircuit, Language
from mpqp.core.instruction.measurement.expectation_value import ExpectationMeasure
from mpqp.execution.job import JobType


def test(bind: MPQPBinding, programSet: bool = True):
    from braket.program_sets import ProgramSet

    if not isinstance(bind.circuits, list):
        circuits = [bind.circuits]
    else:
        circuits = bind.circuits
    translated = []
    for c in circuits:
        if isinstance(c, QCircuit):
            translated.append(c.to_other_language(Language.BRAKET))
        elif isinstance(c, MPQPBinding):
            translation = test(c, False)
            if isinstance(translation, list):
                translated.extend(translation)
            else:
                translated.append(translation)
    var = bind.value
    if var:
        converted = {}
        for key in var.keys():
            val = var[key]
            if any([not isinstance(v, float | int) for v in val]):
                raise ValueError(f"Cannot use complex parameters in Braket. Got: {val}")
            converted.update({str(key): tuple(val)})
        var = converted

    from braket.program_sets import CircuitBinding

    result = []
    for t in translated:
        if var:
            if bind.job_type == JobType.OBSERVABLE and bind.measurements:
                obs = []
                for m in bind.measurements:
                    assert isinstance(m, ExpectationMeasure)
                    obs.extend(
                        [o.to_other_language(Language.BRAKET) for o in m.observables]
                    )
                result.append(CircuitBinding(t, input_sets=var, observables=obs))
            else:
                result.append(CircuitBinding(t, input_sets=var))
        else:
            if bind.job_type == JobType.OBSERVABLE and bind.measurements:
                obs = []
                for m in bind.measurements:
                    assert isinstance(m, ExpectationMeasure)

                    obs.extend(
                        [o.to_other_language(Language.BRAKET) for o in m.observables]
                    )
                if bind.mode == BindingMode.ZIP:
                    i = 0
                    j = 0
                    while j < (len(obs)):
                        if isinstance(bind.circuits[i], MPQPBinding):
                            for k in range(len(bind.circuits[i].circuits)):
                                result.append(CircuitBinding(translated[i + k], obs[j]))
                            i += len(bind.circuits[i].circuits)
                            j += 1
                        else:
                            result.append(CircuitBinding(translated[i], obs[j]))
                            i += 1
                            j += 1
                else:
                    result.append(CircuitBinding(t, observables=obs))
            else:
                result.append(t)
    if programSet:
        from braket.program_sets import ProgramSet

        return ProgramSet(result, bind.shots)
    else:
        return result

In [ ]:
from sympy import Symbol
from mpqp.core.circuit import CircuitBinding as MPQPBinding
from braket.devices import LocalSimulator
from mpqp.core.instruction.measurement.expectation_value import Observable
from mpqp.core.instruction.measurement.pauli_string import pI, pX, pZ
from mpqp.gates import *
from braket.program_sets import ProgramSet, CircuitBinding

t = Symbol("t")
r = Symbol("r")

nb_param = 2
nb_circuits = 2
nb_obs = 3

m = ExpectationMeasure([Observable(pZ), Observable(pX), Observable(pI)], [0])
c1 = QCircuit([Ry(t, 0), Rx(r, 0)])  # .to_other_language(Language.BRAKET)
c = QCircuit([Ry(1, 0), Rx(0.4, 0)])  # .to_other_language(Language.BRAKET)
p = {t: [0, 1], r: [1, 0]}
test = MPQPBinding([MPQPBinding(c1, p)], measurements=[m])


m = [
    Observable(pZ),  # .to_other_language(Language.BRAKET),
    Observable(pI),  # .to_other_language(Language.BRAKET),
]
"""
mb = ProgramSet([c, CircuitBinding(c1, input_sets=p, observables=m)], 100)
mm = ProgramSet.product([c, CircuitBinding(c1, observables=[m[0]])], observables=[m[1]])

device = LocalSimulator()
result_2 = device.run(mb).result()
for r in result_2:  # type: ignore
    print(len(r))
    """

ValueError: your circuit already contains measurements, you cannot have multiple measurements

c : 1

c1 : ([0,1] Z, [0,1] X, [0,1] I : [1,0.01] Z, [1,0.01] X, [1,0.01] I ) = 6
7 total runs.

In [22]:
print(
    ProgramSet.product([c, CircuitBinding(c1, observables=[m[0]])], observables=[m[1]])
)

ValueError: Cannot specify observables in both circuit bindings and product

In [11]:
from mpqp.execution.devices import AWSDevice
from mpqp.execution.runner import run
from mpqp.core.instruction.measurement.expectation_value import Observable
from mpqp.core.instruction.measurement.pauli_string import pI, pX, pZ
from mpqp.gates import *
from mpqp import QCircuit, ExpectationMeasure

x = 0
y = 1
c = QCircuit(
    [Rz(x, 0), Rx(y, 0), Rz(x, 0), ExpectationMeasure(Observable(pX), shots=100)]
)

run(c, AWSDevice.BRAKET_LOCAL_SIMULATOR)

hfzehijfe
[0.46, 0.54]


Result(Job(JobType.OBSERVABLE, QCircuit([Rz(0, 0), Rx(1, 0), Rz(0, 0), ExpectationMeasure(Observable(pX, 'observable_0'), shots=100)]), AWSDevice.BRAKET_LOCAL_SIMULATOR), np.float64(-0.08000000000000002), None, 100)